In [8]:
import os
from langchain_chroma import Chroma
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import SecretStr
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from rag_tutorial.rag_utils import create_rag_system, Judgement, build_rag_chain, create_judge_llm, make_judges, print_wrapped


In [9]:
api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))
if not api_key.get_secret_value():
    raise RuntimeError("OPENROUTER_API_KEY is not set")

In [10]:
gpt = "openai/gpt-4o-mini"
llm = ChatOpenRouter(
    api_key=api_key,
    model=gpt,
    temperature=0,
)

In [11]:
judge_llm = create_judge_llm("openai/gpt-4o-mini")

In [12]:
# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [13]:
# Load and split documents
documents_path = Path(r"c:\Artificial Intelligence\rag_tutorial\documents")

loader = DirectoryLoader(
    str(documents_path),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(documents)

In [15]:
# Create persistent Chroma database (langchain-chroma writes to persist_directory automatically)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="my_collection"
)

### Semantic search

In [16]:
# Load existing database. Semantic search will use this
sematic_vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="my_collection"
)

In [17]:
# semantic retriever
semantic_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

### Hybrid search with Chroma


In [18]:
# Hybrid search with Chroma
# Create BM25 retriever for keyword search
data = vectorstore.get()  # dict with "documents" and "metadatas"
bm25_docs = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(data["documents"], data["metadatas"])
]
bm25_retriever = BM25Retriever.from_documents(bm25_docs) # keyword search
bm25_retriever.k = 5

In [19]:
# Combine retrievers with weights
ensemble_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # 70% semantic, 30% keyword
)

In [20]:
test_set = [
    {"question": "Was ist Overfitting?",
     "reference": "Overfitting tritt auf, wenn ein Modell die Trainingsdaten "
     "auswendig lernt und nicht auf neue Daten generalisiert."},
    {"question": "Wie werden Neuronale Netze trainiert?",
     "reference": "Neuronale Netze werden durch Backpropagation und Gradient "
     "Descent trainiert."},
    {"question": "Was ist der Vorteil von Random Forests?",
     "reference": "Random Forests kombinieren viele Decision Trees und sind "
     "oft genauer als einzelne Modelle."},
    {"question": "Wie hilft RAG bei LLMs?",
     "reference": "RAG hilft LLMs, präziser mit spezifischem Wissen zu antworten."},
    {"question": "Was ist der Unterschied zwischen Bagging und Boosting?",
     "reference": "Diese Information ist nicht in den Dokumenten enthalten."},
]

In [21]:
judge_faithfulness, judge_relevance, judge_correctness = make_judges(judge_llm)

In [22]:
messages = [
    ("system", "You are an AI assistant that can answer questions about the history of human civilization. "
    "You are given a question and a list of documents and need to answer the question. "
    "Answer the question only based on these documents. These documents can help you answer the question: "
    "{context}. If you are not sure about the answer, you can say 'I don't know' or 'I don't know the answer to that question.'"),
    ("human", "{question}"),
]
prompt = ChatPromptTemplate.from_messages(messages=messages)
chain = prompt | llm | StrOutputParser()

In [23]:
def evaluate(k):
    rows = []
    for item in test_set:
        q, ref = item["question"], item["reference"]
        context = "\n\n".join(d.page_content for d in ensemble_retriever.invoke(q))
        answer = chain.invoke({"context": context, "question": q})
        rows.append({
            "question": q,
            "answer": answer,
            "faithful": judge_faithfulness(answer, context).verdict,
            "relevant": judge_relevance(q, answer).verdict,
            "correct": judge_correctness(q, ref, answer).verdict,
        })
    return pd.DataFrame(rows)

In [24]:
results_k2 = evaluate(k=2)

In [25]:
results_k2[["question", "faithful", "relevant", "correct"]]

,question,faithful,relevant,correct
0,Was ist Overfitting?,True,True,True
1,Wie werden Neuronale Netze trainiert?,True,True,False
2,Was ist der Vorteil von Random Forests?,True,True,False
3,Wie hilft RAG bei LLMs?,True,False,False
4,Was ist der Unterschied zwischen Bagging und B...,True,True,False
